# Q-Mini — Separate Training Runner (Resume-safe)

This notebook is the **only training activation path** for a package created by QMini_Training_Ready_Build.ipynb.

The model builder has already stored:
- random-initialized Q2-v2 model weights;
- the matching tokenizer and prepared train/validation token streams;
- the complete Q-Mini training source under `training_runtime/`;
- a dormant training state with zero optimizer steps.

Training is deliberately OFF by default. To start, review the settings cell and explicitly set `START_TRAINING = True`.

## Interruption and recovery

The trainer first writes its complete resumable checkpoint to Colab's local disk. At every configured save interval, it copies an immutable checkpoint generation plus matching metrics and run contract to Drive. Drive copies are hash-checked and the PyTorch archive CRC is checked before the single `training_progress.json` record is advanced. Two verified generations are retained. On a new Colab session, the runner restores the newest complete generation and falls back to the previous generation if the newest is damaged.

The visible progress number is replaced after each verified save; the recovery files are retained separately. A partial copy never becomes the current checkpoint.

**Important limit:** an interruption between save intervals can lose work since the last verified save. The default interval is 10 optimizer steps. No cloud notebook can guarantee zero lost work if the runtime or Drive is abruptly disconnected.

In [ ]:
# ---------------- USER SETTINGS ----------------
PACKAGE_NAME_OVERRIDE = None  # exact folder name under MyDrive/QMini-Light-0.7M/training_ready, or None for newest package

START_TRAINING = False         # set True only after package/GPU checks pass
TARGET_OPTIMIZER_STEPS = 20000 # total optimizer-step target; resume continues toward this number
BATCH_SIZE = 1
SEQUENCE_LENGTH = 512
GRAD_ACCUM = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY = 10          # verified Drive checkpoint interval
EVALUATE_EVERY = 100
EVALUATION_BATCHES = 8
MIN_FREE_GPU_GIB = 8.0

# A lease heartbeat is refreshed every 45 seconds. An abandoned lease is
# automatically recoverable after three minutes without a heartbeat.
LEASE_STALE_SECONDS = 180
LEASE_HEARTBEAT_SECONDS = 45

print("Training activation:", START_TRAINING)
print("Target optimizer steps:", TARGET_OPTIMIZER_STEPS)
print("Checkpoint interval:", CHECKPOINT_EVERY)

In [ ]:
# ---------------- MOUNT DRIVE AND SELECT THE BUILT PACKAGE ----------------
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import shutil
import sys
import time
import uuid

MY_DRIVE = Path("/content/drive/MyDrive")
PACKAGE_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "training_ready"
if not PACKAGE_ROOT.is_dir():
    raise FileNotFoundError(
        f"No training-ready Q-Mini package folder: {PACKAGE_ROOT}. "
        "Run QMini_Training_Ready_Build.ipynb first."
    )

if PACKAGE_NAME_OVERRIDE:
    PACKAGE_DIR = PACKAGE_ROOT / PACKAGE_NAME_OVERRIDE
    if not PACKAGE_DIR.is_dir():
        raise FileNotFoundError(f"Selected Q-Mini package not found: {PACKAGE_DIR}")
else:
    candidates = [
        path for path in PACKAGE_ROOT.glob("QMini-Light-0.7M-Q2v2-*")
        if (path / "training_runtime" / "TRAINING_RUNTIME.json").is_file()
    ]
    if not candidates:
        raise FileNotFoundError("No complete Q-Mini training-ready package was found.")
    candidates.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    PACKAGE_DIR = candidates[0]

INITIAL_DIR = PACKAGE_DIR / "initial_brain"
DATASET_DIR = PACKAGE_DIR / "dataset"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"
CONFIG_PATH = INITIAL_DIR / "config.json"
INITIAL_MODEL_PATH = INITIAL_DIR / "model.qmini.npz"
RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
RUNTIME_MANIFEST_PATH = RUNTIME_DIR / "TRAINING_RUNTIME.json"
TRAINING_DEFAULTS_PATH = RUNTIME_DIR / "TRAINING_DEFAULTS.json"
if not TRAINING_DEFAULTS_PATH.is_file():
    raise FileNotFoundError(f"Bundled training defaults are missing: {TRAINING_DEFAULTS_PATH}")
MIRROR_DIR = PACKAGE_DIR / "training_state"
PROGRESS_PATH = MIRROR_DIR / "training_progress.json"

LOCAL_RUN_DIR = Path("/content/qmini-light-0.7m-local-runs") / PACKAGE_DIR.name
TRAIN_BIN = DATASET_DIR / "train.bin"
VALID_BIN = DATASET_DIR / "valid.bin"
TRAIN_MASK = DATASET_DIR / "train.mask.bin"
VALID_MASK = DATASET_DIR / "valid.mask.bin"

print("Selected package:", PACKAGE_DIR)
print("Initial Q2 model:", INITIAL_MODEL_PATH)
print("Persistent training state:", MIRROR_DIR)
print("Local working checkpoint:", LOCAL_RUN_DIR)

In [ ]:
# ---------------- INSTALL RUNTIME DEPENDENCIES + VERIFY PACKAGE ----------------
# Preserve Colab's preinstalled CUDA PyTorch wheel; never replace it blindly.
import subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "numpy>=1.24", "tokenizers>=0.15", "packaging>=23",
], check=True)

import torch
from packaging.version import Version
if Version(torch.__version__.split("+", 1)[0]) < Version("2.10.0"):
    raise RuntimeError(f"Q-Mini requires PyTorch >=2.10.0; found {torch.__version__}.")
if not torch.cuda.is_available():
    raise RuntimeError(
        "The training runner requires a CUDA GPU. Model building is a separate CPU-compatible step."
    )

gpu_name = torch.cuda.get_device_name(0)
free_gpu_bytes, total_gpu_bytes = torch.cuda.mem_get_info(0)
print("PyTorch:", torch.__version__)
print("Training GPU:", gpu_name)
print(f"GPU memory: {free_gpu_bytes/1024**3:.2f} GiB free / {total_gpu_bytes/1024**3:.2f} GiB total")

if free_gpu_bytes < MIN_FREE_GPU_GIB * (1024 ** 3):
    raise RuntimeError(
        f"Training is blocked: free GPU memory is below {MIN_FREE_GPU_GIB:.1f} GiB. "
        "Use a larger-memory GPU; lowering this guard does not guarantee a successful run."
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

runtime_manifest = json.loads(RUNTIME_MANIFEST_PATH.read_text(encoding="utf-8"))
training_defaults = json.loads(TRAINING_DEFAULTS_PATH.read_text(encoding="utf-8"))
if training_defaults.get("format") != "qmini-training-defaults-v1" or training_defaults.get("enabled") is not False:
    raise RuntimeError("Bundled training defaults are missing or unexpectedly enabled.")
if runtime_manifest.get("format") != "qmini-training-runtime-v1":
    raise RuntimeError("Unsupported bundled Q-Mini training runtime.")
if runtime_manifest.get("enabled") is not False or runtime_manifest.get("requires_manual_activation") is not True:
    raise RuntimeError("Training runtime activation contract is invalid.")
if not (RUNTIME_DIR / "qmini" / "checkpointing.py").is_file():
    raise FileNotFoundError("Bundled checkpoint recovery module is missing.")
for record in runtime_manifest.get("source_files", []):
    relative = Path(record["path"])
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError("Training runtime manifest contains an unsafe path.")
    source = RUNTIME_DIR / relative
    if not source.is_file() or sha256_file(source) != record.get("sha256"):
        raise RuntimeError(f"Bundled training source hash mismatch: {relative}")

# Import the exact source code embedded with this model package, not a later
# GitHub checkout that might silently change the training algorithm.
sys.path.insert(0, str(RUNTIME_DIR))
from qmini.config import QMiniConfig
from qmini.export import verify_packed_export
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS
from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
)
from qmini.checkpointing import (
    TrainingSessionLease,
    mark_training_status,
    restore_latest_checkpoint,
)

config = QMiniConfig.from_json(CONFIG_PATH)
if config.max_context != 512:
    raise RuntimeError(
        "This Light runner requires a 512-token model package. Rebuild it with the updated "
        "QMini_Training_Ready_Build.ipynb notebook before training."
    )
packaged_sequence_length = training_defaults.get("sequence_length")
if (
    isinstance(packaged_sequence_length, bool)
    or not isinstance(packaged_sequence_length, int)
    or packaged_sequence_length != config.max_context
):
    raise RuntimeError("Bundled training sequence length must match the 512-token model context.")
model_manifest = json.loads((INITIAL_DIR / "model_manifest.json").read_text(encoding="utf-8"))
if model_manifest.get("training_runtime_manifest_sha256") != sha256_file(RUNTIME_MANIFEST_PATH):
    raise RuntimeError("Bundled training runtime manifest fingerprint differs from the initial-model manifest.")
if model_manifest.get("training_steps") != 0 or model_manifest.get("optimizer_steps") != 0:
    raise RuntimeError("The selected initial model is not a step-zero model.")
if model_manifest.get("training_runtime_enabled") is not False:
    raise RuntimeError("The selected model package has an unexpected active training runtime.")
_verify_initial_brain_artifact(INITIAL_MODEL_PATH, config)

tokenizer = load_tokenizer(TOKENIZER_PATH)
if tokenizer.get_vocab_size() != config.vocab_size:
    raise RuntimeError("Tokenizer vocabulary differs from the model configuration.")
for token in SPECIAL_TOKENS:
    if tokenizer.token_to_id(token) is None:
        raise RuntimeError(f"Tokenizer is missing required control token {token!r}.")

dataset_manifest = _verify_prepared_dataset_manifest(
    TRAIN_BIN, VALID_BIN, TOKENIZER_PATH,
    TRAIN_MASK, VALID_MASK, config.vocab_size,
)
train_tokens = _load_tokens(TRAIN_BIN, "Training")
valid_tokens = _load_tokens(VALID_BIN, "Validation")
_validate_token_ids(train_tokens, config.vocab_size, "Training")
_validate_token_ids(valid_tokens, config.vocab_size, "Validation")
if TRAIN_MASK.stat().st_size != TRAIN_BIN.stat().st_size // 2:
    raise RuntimeError("Training loss-mask file has an invalid length.")
if VALID_MASK.stat().st_size != VALID_BIN.stat().st_size // 2:
    raise RuntimeError("Validation loss-mask file has an invalid length.")

if not 2 <= SEQUENCE_LENGTH <= config.max_context:
    raise ValueError("SEQUENCE_LENGTH is outside the model's configured context.")
if TARGET_OPTIMIZER_STEPS < 1 or CHECKPOINT_EVERY < 1 or EVALUATE_EVERY < 1:
    raise ValueError("Training steps and checkpoint/evaluation intervals must be positive.")
if BATCH_SIZE < 1 or GRAD_ACCUM < 1 or EVALUATION_BATCHES < 1:
    raise ValueError("Batch size, gradient accumulation and evaluation batches must be positive.")

print("Model parameters:", f"{config.estimated_parameter_count():,}")
print("Tokenizer vocabulary:", config.vocab_size)
# Conservative checkpoint-storage preflight. A QAT checkpoint contains
# FP32 model parameters plus AdamW moments; two recovery generations are kept
# and a third temporary generation exists while a new one is uploaded.
checkpoint_estimate_bytes = int(config.estimated_parameter_count() * 12 * 1.2 + 512 * 1024**2)
drive_required_bytes = 3 * checkpoint_estimate_bytes + 1024**3
local_required_bytes = 2 * checkpoint_estimate_bytes + 1024**3
drive_free_bytes = shutil.disk_usage(MY_DRIVE).free
local_free_bytes = shutil.disk_usage("/content").free
print(f"Estimated checkpoint size: {checkpoint_estimate_bytes/1024**3:.2f} GiB")
print(f"Drive free: {drive_free_bytes/1024**3:.2f} GiB | conservative required during upload: {drive_required_bytes/1024**3:.2f} GiB")
print(f"Colab local free: {local_free_bytes/1024**3:.2f} GiB | required for atomic local save: {local_required_bytes/1024**3:.2f} GiB")
if drive_free_bytes < drive_required_bytes:
    raise RuntimeError("Insufficient Google Drive free space for two recovery checkpoints plus an in-progress upload.")
if local_free_bytes < local_required_bytes:
    raise RuntimeError("Insufficient Colab /content disk space for atomic checkpoint creation.")

print("Train tokens:", f"{len(train_tokens):,}")
print("Validation tokens:", f"{len(valid_tokens):,}")
print("Training mask:", TRAIN_MASK)
print("Runtime source files verified:", len(runtime_manifest["source_files"]))
print("Model weights are still at optimizer step zero until training is explicitly started.")

In [ ]:
# ---------------- DISPLAY THE CURRENT DRIVE PROGRESS ----------------
# Do not restore or rewrite checkpoint state here. The actual restore happens
# only after the training writer lease is acquired in the activation cell.
MIRROR_DIR.mkdir(parents=True, exist_ok=True)
restored = None

if PROGRESS_PATH.is_file():
    current_progress = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
    print("Drive training status:", current_progress.get("status"))
    print(
        "Last verified optimizer step:",
        current_progress.get("current_step", 0),
        "/",
        current_progress.get("target_steps"),
    )
    print("Tokens seen:", f"{int(current_progress.get('tokens_seen', 0)):,}")
else:
    print("No Drive training progress record exists. Run the model builder first.")

print("No checkpoint restore or optimizer update has run in this cell.")

In [ ]:
# ---------------- EXPLICIT TRAINING ACTIVATION ----------------
if not START_TRAINING:
    print("TRAINING IS OFF. Set START_TRAINING = True in the settings cell to activate it.")
    print("No optimizer was created and no model weight was changed by this runner.")
else:
    from qmini.train import train_model

    LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
    session_started = datetime.now(timezone.utc).isoformat()
    print("Starting Q-Mini training session:", session_started)
    print("Optimizer step target:", TARGET_OPTIMIZER_STEPS)
    print("Checkpoint interval:", CHECKPOINT_EVERY, "steps")

    # Acquire the Drive writer lease BEFORE restoring the latest checkpoint.
    # This prevents a stale restore from another notebook session from being
    # published over a newer progress counter.
    with TrainingSessionLease(
        MIRROR_DIR,
        stale_after_seconds=LEASE_STALE_SECONDS,
        heartbeat_seconds=LEASE_HEARTBEAT_SECONDS,
    ) as lease:
        previous_progress = (
            json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
            if PROGRESS_PATH.is_file() else {}
        )
        if previous_progress.get("status") == "running":
            mark_training_status(
                MIRROR_DIR,
                "interrupted",
                note="A previous Colab runtime ended without releasing its training lease.",
            )

        # Re-read and restore while holding the lease; the earlier display-only
        # restore in the preceding cell is not trusted for the actual training run.
        restored = restore_latest_checkpoint(MIRROR_DIR, LOCAL_RUN_DIR)
        if restored is None:
            current_progress = (
                json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
                if PROGRESS_PATH.is_file() else {}
            )
            if current_progress.get("current_step", 0) not in (0, None):
                raise RuntimeError(
                    "Drive progress says training has advanced, but no verified checkpoint "
                    "generation can be restored. Refusing to restart from step zero."
                )
            resume_run = False
            extend_schedule = False
            current_step = 0
        else:
            resume_run = True
            extend_schedule = TARGET_OPTIMIZER_STEPS > restored["target_steps"]
            current_step = restored["step"]
            if current_step > TARGET_OPTIMIZER_STEPS:
                raise ValueError(
                    "TARGET_OPTIMIZER_STEPS cannot be below the restored optimizer step."
                )
            if restored["target_steps"] > TARGET_OPTIMIZER_STEPS and current_step < TARGET_OPTIMIZER_STEPS:
                raise ValueError(
                    "TARGET_OPTIMIZER_STEPS is below the previous run's total target. "
                    "Raise the target or use a new, separately fingerprinted run."
                )

        mark_training_status(
            MIRROR_DIR,
            "running",
            note=f"Colab training session started at {session_started}.",
        )
        print("Resume:", resume_run, "| extend schedule:", extend_schedule)
        print("Starting optimizer step:", current_step)
        print("Optimizer step target:", TARGET_OPTIMIZER_STEPS)

        FINAL_LOCAL_DIR = train_model(
            config=config,
            train_bin=TRAIN_BIN,
            valid_bin=VALID_BIN,
            output_dir=LOCAL_RUN_DIR,
            steps=TARGET_OPTIMIZER_STEPS,
            batch_size=BATCH_SIZE,
            seq_len=SEQUENCE_LENGTH,
            grad_accum=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            eval_every=EVALUATE_EVERY,
            save_every=CHECKPOINT_EVERY,
            eval_batches=EVALUATION_BATCHES,
            resume=resume_run,
            tokenizer_path=TOKENIZER_PATH,
            extend_schedule=extend_schedule,
            train_mask_bin=TRAIN_MASK,
            valid_mask_bin=VALID_MASK,
            init_from_initial=None if resume_run else INITIAL_MODEL_PATH,
            checkpoint_mirror_dir=MIRROR_DIR,
            checkpoint_session_id=lease.session_id,
        )

        # Verify and publish the trained deployment export separately from the
        # resumable optimizer checkpoint. The step-zero initial model is immutable.
        final_manifest_path = FINAL_LOCAL_DIR / "model_manifest.json"
        final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
        final_packed_path = FINAL_LOCAL_DIR / final_manifest["packed_export"]
        final_report = verify_packed_export(final_packed_path)
        if (
            final_report.get("format") != "qmini-packed-v2"
            or final_report.get("q2_version") != 2
            or final_report.get("parameter_count") != config.estimated_parameter_count()
        ):
            raise RuntimeError("Final trained Q2-v2 export failed verification.")
        if sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256"):
            raise RuntimeError("Final trained model SHA-256 differs from its manifest.")

        final_step = int(final_manifest["step"])
        trained_root = PACKAGE_DIR / "trained_models"
        trained_root.mkdir(parents=True, exist_ok=True)
        final_name = f"QMini-Light-0.7M-trained-step-{final_step:012d}"
        final_drive_dir = trained_root / final_name
        staging_dir = trained_root / f".{final_name}.staging-{uuid.uuid4().hex[:8]}"

        if final_drive_dir.exists():
            existing_manifest = json.loads(
                (final_drive_dir / "model_manifest.json").read_text(encoding="utf-8")
            )
            existing_packed = final_drive_dir / existing_manifest["packed_export"]
            if (
                existing_manifest.get("step") != final_step
                or existing_manifest.get("packed_export_sha256") != final_manifest.get("packed_export_sha256")
                or sha256_file(existing_packed) != existing_manifest.get("packed_export_sha256")
            ):
                raise FileExistsError(
                    f"An existing trained model conflicts with this step: {final_drive_dir}"
                )
            verify_packed_export(existing_packed)
        else:
            try:
                shutil.copytree(FINAL_LOCAL_DIR, staging_dir)
                copied_manifest = json.loads((staging_dir / "model_manifest.json").read_text(encoding="utf-8"))
                copied_packed = staging_dir / copied_manifest["packed_export"]
                if sha256_file(copied_packed) != copied_manifest.get("packed_export_sha256"):
                    raise RuntimeError("Drive final-model copy failed SHA-256 verification.")
                verify_packed_export(copied_packed)
                staging_dir.rename(final_drive_dir)
            except Exception:
                shutil.rmtree(staging_dir, ignore_errors=True)
                raise

        directory_digest = hashlib.sha256()
        for path in sorted(final_drive_dir.rglob("*")):
            if path.is_file():
                directory_digest.update(path.relative_to(final_drive_dir).as_posix().encode("utf-8"))
                directory_digest.update(sha256_file(path).encode("ascii"))
        final_model_digest = directory_digest.hexdigest()

        progress = mark_training_status(
            MIRROR_DIR,
            "completed",
            final_model_sha256=final_model_digest,
            note=f"Final Q2-v2 deployment export stored at trained_models/{final_name}",
        )

    print("\nQ-MINI TRAINING SESSION COMPLETED")
    print("Current optimizer step:", progress["current_step"], "/", progress["target_steps"])
    print("Tokens seen:", f"{progress['tokens_seen']:,}")
    print("Verified final model:", final_drive_dir)
    print("Progress record (single current step):", PROGRESS_PATH)
    print("Resumable checkpoints (latest two):", MIRROR_DIR)

## What happens after an interruption?

1. Each checkpoint is first completed on Colab's local disk.
2. The checkpoint, metrics log and run contract are copied to uniquely named Drive generations.
3. SHA-256 and PyTorch archive CRC are verified on the Drive copy.
4. Only then are `latest_checkpoint.json` and the one-line `training_progress.json` record advanced.
5. The last two verified generations remain available. An incomplete staging file is ignored.
6. A new Colab session restores the newest valid generation, or falls back to the previous valid one.

The progress record shows one current value, for example `current_step: 1250` and `target_steps: 20000`; it is replaced after the next verified checkpoint. The recovery snapshots are separate from this display record.

An automatic recovery can only restore the last verified checkpoint. Work since that save interval may need to be repeated. Drive outages during upload stop the current run rather than falsely advancing the saved step.

The training runtime is bundled with the model but not embedded as a running background process. It is activated only by this separate notebook.